# E17 — Qwen3 EN→BN probe (open weights, deployable)

| Candidate | n | Token F1 | vs Google |
|---|---|---|---|
| **Claude Opus 5** | 24 | **0.6865** | **+0.0631** (t 5.88) |
| Codex (GPT-5) | 10 | 0.6483 | +0.0253 |
| Google draft | — | 0.5918 *(200 rows)* | baseline |
| NLLB-200 1.3B | 200 | 0.5480 | **−0.0439** (t −11.41) ❌ |

## The question this run answers

Claude wins — but there is **no Claude API here**, so its +0.0631 cannot be deployed across
107,737 rows. NLLB showed the whole dedicated-MT category *loses*. So:

> **Can an open-weights model we can actually run at scale beat Google's 0.5918?**

If yes, the full re-translation is a fleet job with `Qwen3-235B-A22B`. If no, E17 closes and
effort moves to E05 (train longer).

## 🔴 Why 14B and not 235B
`Qwen3-235B-A22B` needs **~470 GB** in bf16, **~120 GB** at 4-bit. Kaggle's ceiling is **32 GB**
(2×T4). It is not a tuning problem — it is off by 15×. This run is the **deployability signal**;
the 235B test belongs on the unlimited-GPU box via `run_local_models.py`.

Qwen3 is the right family regardless of size: it expanded to **119 languages including Bengali**,
where Qwen2.5's headline list does not feature it.

In [ ]:
# 1 ── hardware gate
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
assert torch.cuda.is_available(), "no GPU — enable the accelerator"
cap = torch.cuda.get_device_capability()
N_GPU = torch.cuda.device_count()
TOTAL = sum(torch.cuda.get_device_properties(i).total_memory for i in range(N_GPU)) / 1e9
print(f"{N_GPU}× {torch.cuda.get_device_name(0)} | sm_{cap[0]}{cap[1]} | {TOTAL:.1f} GB total")
assert cap[0] >= 7, f"sm_{cap[0]}{cap[1]} unsupported — Kaggle's torch ships no sm_60 kernels (P100)"
BF16 = cap[0] >= 8      # T4 is sm_75: bf16 is EMULATED and slower than fp16, not hardware
print(f"bf16 hardware: {BF16} -> compute dtype {'bfloat16' if BF16 else 'float16'}")

In [ ]:
# 2 ── deps. Not pinning to 4.57.3 here: Qwen3 needs a transformers new enough to
#      know its architecture. This notebook trains nothing, so the T5 pin is irrelevant.
!pip install -q -U transformers accelerate bitsandbytes
import transformers, bitsandbytes
print("transformers", transformers.__version__, "| bitsandbytes", bitsandbytes.__version__)

In [ ]:
# 3 ── data
import glob, pandas as pd
N_ROWS = 200
src = glob.glob("/kaggle/input/**/PROBE_200_dev.csv", recursive=True)
assert src, "attach the nascenia-e17-probe dataset"
rows = pd.read_csv(src[0]).head(N_ROWS)
w = rows.english.astype(str).str.split().str.len()
print(f"{src[0]} | {len(rows)} rows | english words mean {w.mean():.0f} p95 {w.quantile(.95):.0f}")

In [ ]:
# 4 ── load Qwen3-14B in 4-bit NF4
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = "Qwen/Qwen3-14B"
tok = AutoTokenizer.from_pretrained(MODEL)
# 🔴 decoder-only models MUST left-pad for batched generation, or short prompts
# get right-padding between the prompt and the first generated token and the
# output is garbage.
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if BF16 else torch.float16,
    ),
    device_map="auto",
).eval()
print(f"{MODEL} loaded | {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# 5 ── prompt: byte-identical to run_local_models.py so the fleet run is comparable
PROMPT = (
    "Translate this English doctor's reply into Bengali.\n\n"
    "Rules:\n"
    "- Faithful, plain, sentence-by-sentence. Do NOT improve, polish, shorten, "
    "expand, restructure, or fix the author's grammar.\n"
    "- Keep medical abbreviations in Latin script (PCOD, LH/FSH, CA 125, MRI, PFT).\n"
    "- Do NOT add a greeting, sign-off, branding, or any commentary.\n"
    "- Output ONLY the Bengali translation, nothing else.\n\n"
    "English:\n{en}\n\nBengali:"
)

def build(en):
    return tok.apply_chat_template(
        [{"role": "user", "content": PROMPT.format(en=en)}],
        tokenize=False, add_generation_prompt=True,
        enable_thinking=False,      # translation is not a reasoning task; also avoids <think> blocks
    )

print(build("The patient has a fever.")[:400])

In [ ]:
# 6 ── smoke test before committing an hour
import re, time

def clean(t):
    t = re.sub(r"<think>.*?</think>", "", t, flags=re.S)      # belt-and-braces
    return t.strip().strip('"').strip()

def generate(batch_en, max_new=768):
    enc = tok([build(e) for e in batch_en], return_tensors="pt",
              padding=True, truncation=True, max_length=1536).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                             pad_token_id=tok.pad_token_id)
    gen = out[:, enc["input_ids"].shape[1]:]      # strip the prompt
    return [clean(x) for x in tok.batch_decode(gen, skip_special_tokens=True)]

t0 = time.time()
s = generate([rows.english.iloc[0]])[0]
print(f"smoke ({time.time()-t0:.0f}s):\n{s[:400]}")
assert s.strip(), "❌ empty output"
assert any("\u0980" <= c <= "\u09FF" for c in s), "❌ output contains no Bengali characters"

In [ ]:
# 7 ── translate all rows, length-sorted, OOM-tolerant, saving as it goes
BATCH = 4
order = sorted(range(len(rows)), key=lambda i: len(str(rows.english.iloc[i])))
got, t0, i = {}, time.time(), 0

while i < len(order):
    bs, idx = BATCH, order[i:i+BATCH]
    while True:
        try:
            out = generate([rows.english.iloc[j] for j in idx]); break
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); bs //= 2
            assert bs >= 1, "OOM at batch 1 — drop to Qwen3-8B"
            idx = idx[:bs]; print(f"    ⚠️ OOM -> batch {bs}", flush=True)
    got.update(dict(zip(idx, out)))
    i += len(idx)
    el = time.time() - t0
    print(f"  {i}/{len(order)}  {el/60:.1f} min (eta {el/i*(len(order)-i)/60:.1f} min)", flush=True)
    if i % 40 == 0 or i >= len(order):      # partial save — a late crash loses nothing
        pd.DataFrame({"hcm_id": [rows.hcm_id.iloc[k] for k in sorted(got)],
                      "bengali": [got[k] for k in sorted(got)]}
                     ).to_csv("qwen3_14b_TRANSLATED.csv", index=False, encoding="utf-8")
print(f"\ndone in {(time.time()-t0)/60:.1f} min")

In [ ]:
# 8 ── verify + write
bn = [got[k] for k in range(len(rows))]
assert len(bn) == len(rows), f"❌ {len(bn)} outputs for {len(rows)} rows"
assert all(str(x).strip() for x in bn), "❌ blank translation"

out = pd.DataFrame({"hcm_id": rows.hcm_id.values, "bengali": bn})
out.to_csv("qwen3_14b_TRANSLATED.csv", index=False, encoding="utf-8")
s = out.bengali.astype(str)

bengali_frac = s.apply(lambda x: sum('\u0980' <= c <= '\u09FF' for c in x) / max(len(x), 1))
print(f"✅ qwen3_14b_TRANSLATED.csv | {len(out)} rows | mean {s.str.split().str.len().mean():.0f} words")
print(f"   Bengali char fraction: mean {bengali_frac.mean():.2f} "
      f"| rows below 0.5: {(bengali_frac < 0.5).sum()}  (should be 0)")
print("\n🔴 CONTAMINATION — both must be 0.0%; an LLM that notices the target style")
print("   and adds these scores higher for the WRONG reason.")
print(f"   হেলো {s.str.startswith('হেলো').mean()*100:.1f}% (target 76.4%) · "
      f"নাসেনিয়া {s.str.contains('নাসেনিয়া').mean()*100:.1f}% (target 50.0%)")
print(f"\nsample: {out.bengali.iloc[0][:250]}")
print("\nDownload into E17_RETRANSLATE/, then: python score_all.py")